In [1]:
from bbs_iss.entities.issuer import IssuerInstance

# Deploying Issuer Instance
issuer = IssuerInstance()
pub_key = issuer.public_key
print(pub_key.key)


b'\x82\xef\xfa\xa8\xf3`R\xec\xaf\xc4\\\x1cW\xfb\x8a?@*\x0b\x8d\xa26\xcd\xfeP\xf3Ajn\xa4/\xbbC\xb4n\xa7\x06\xa9\xfd\x8a\x12ESx\xa5JXU\x13\xdb\xd3\xb8\x08JN\xbc\x81\x83ux\xb0_ \xac\xb3\xf9\xbf\xe4\x14s\xc0\xa2\x9eR\xf4\x8b\r\x93\x13M\xf1\xe0\xffBW\xa9\xd97\x9cs\xb8J\x08%\xd3\x94'


In [ ]:
from bbs_iss.interfaces.requests_api import api
import os

# Initiating Issuance
issuance_request = api.VCIssuanceRequest()

# Sending Request to Issuer and Receiving Freshness
nonce = issuer.process_request(issuance_request)

# Building Blind Sign Request
attributes = api.IssuanceAttributes()
link_secret = os.urandom(32)
attributes.append(link_secret, api.AttributeType.HIDDEN)
for i in range(10):
    attributes.append(f"attribute{i}")
blind_sign_request = api.BlindSignRequest(attributes, nonce, public_key=pub_key)

# Sending Blind Sign Request to Issuer and Receiving Blind Signature
blind_signature = issuer.process_request(blind_sign_request)




FreshnessValueError: Invalid freshness value

In [ ]:
import ursa_bbs_signatures as bbs

# Unblinding Signature
unblinded_signature = bbs.unblind_signature(bbs.UnblindSignatureRequest(
    blinded_signature=blind_signature,
    blinding_factor=attributes.get_blinding_factor()
))

# Verifying Validity
ver_request = bbs.VerifyRequest(
    key_pair=bbs.BlsKeyPair(public_key=pub_key.key),
    signature=unblinded_signature,
    messages=attributes.attributes_to_list()
)

bbs.verify(ver_request)